# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso

import urllib.request
import warnings
warnings.filterwarnings('ignore')

## 2. Datos

In [ ]:
df = pd.read_csv("./data/train.csv")

### 2.1 Exploración de los datos

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# comprobamos si hay nulos
df.isnull().sum()

No hay nulos, eso está bien

In [ ]:
df.describe()

In [ ]:
# miramos el target
df['Price_in_euros'].hist(bins=30)
plt.xlabel('Precio (€)')
plt.ylabel('Frecuencia')
plt.title('Distribución del precio')
plt.show()

El precio está un poco sesgado hacia la derecha, hay algunos portátiles muy caros

In [ ]:
# veamos las columnas categóricas
print("Compañías:", df['Company'].nunique())
print("Tipos:", df['TypeName'].nunique())
print("Sistemas Operativos:", df['OpSys'].nunique())

In [ ]:
df['Company'].value_counts()

In [ ]:
df['TypeName'].value_counts()

## 3. Procesado de datos

Veo que hay varias columnas que tienen texto pero en realidad son números:
- Ram: tiene "8GB", "16GB"... hay que quitar el "GB"
- Weight: tiene "1.86kg"... hay que quitar el "kg"
- Memory: es mas complicado, tiene "256GB SSD", "1TB HDD"...

También hay columnas con mucha info como Cpu, Gpu y ScreenResolution que podemos intentar extraer algo útil

### 3.1 Función para procesar los datos

Creo una función para poder aplicar el mismo procesamiento a train y test

In [ ]:
def procesar_datos(df):
    """
    Procesa el dataframe de portátiles para convertir las columnas 
    de texto a numéricas y crear nuevas features
    """
    df = df.copy()
    
    # --- RAM ---
    # quitamos el "GB" y convertimos a número
    df['Ram'] = df['Ram'].str.replace('GB', '').astype(int)
    
    # --- WEIGHT ---
    # quitamos el "kg" y convertimos a número
    df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)
    
    # --- MEMORY ---
    # esto es más complicado porque hay cosas como "128GB SSD", "1TB HDD", "256GB SSD + 1TB HDD"
    # voy a crear varias columnas
    
    # si tiene SSD
    df['tiene_SSD'] = df['Memory'].str.contains('SSD').astype(int)
    
    # si tiene HDD
    df['tiene_HDD'] = df['Memory'].str.contains('HDD').astype(int)
    
    # si tiene Flash Storage (los mac)
    df['tiene_Flash'] = df['Memory'].str.contains('Flash').astype(int)
    
    # extraer la capacidad total (aproximada)
    def extraer_capacidad(mem):
        total = 0
        # buscar TB y convertir a GB
        if 'TB' in mem:
            # puede ser 1TB o 2TB
            import re
            tb = re.findall(r'(\d+)TB', mem)
            for t in tb:
                total += int(t) * 1000
        # buscar GB
        if 'GB' in mem:
            import re
            gb = re.findall(r'(\d+)GB', mem)
            for g in gb:
                total += int(g)
        return total
    
    df['Capacidad_GB'] = df['Memory'].apply(extraer_capacidad)
    
    # --- SCREEN RESOLUTION ---
    # extraer la resolución y si es touchscreen
    df['es_Touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)
    df['es_IPS'] = df['ScreenResolution'].str.contains('IPS').astype(int)
    
    # extraer resolución (el último valor que tiene formato NUMxNUM)
    def extraer_resolucion(sr):
        import re
        match = re.search(r'(\d{3,4})x(\d{3,4})', sr)
        if match:
            return int(match.group(1)) * int(match.group(2))
        return 0
    
    df['Pixeles'] = df['ScreenResolution'].apply(extraer_resolucion)
    
    # --- CPU ---
    # extraer marca del procesador
    df['CPU_Intel'] = df['Cpu'].str.contains('Intel').astype(int)
    df['CPU_AMD'] = df['Cpu'].str.contains('AMD').astype(int)
    
    # extraer si es i3, i5, i7
    df['CPU_i3'] = df['Cpu'].str.contains('i3').astype(int)
    df['CPU_i5'] = df['Cpu'].str.contains('i5').astype(int)
    df['CPU_i7'] = df['Cpu'].str.contains('i7').astype(int)
    
    # extraer velocidad del procesador (GHz)
    def extraer_ghz(cpu):
        import re
        match = re.search(r'(\d+\.?\d*)GHz', cpu)
        if match:
            return float(match.group(1))
        return 0
    
    df['CPU_GHz'] = df['Cpu'].apply(extraer_ghz)
    
    # --- GPU ---
    # marca de la gráfica
    df['GPU_Nvidia'] = df['Gpu'].str.contains('Nvidia').astype(int)
    df['GPU_AMD'] = df['Gpu'].str.contains('AMD').astype(int)
    df['GPU_Intel'] = df['Gpu'].str.contains('Intel').astype(int)
    
    # --- COMPANY ---
    # las marcas premium suelen ser más caras
    marcas_premium = ['Apple', 'Microsoft', 'Razer', 'MSI']
    df['es_Premium'] = df['Company'].isin(marcas_premium).astype(int)
    
    return df

In [ ]:
# aplicamos el procesamiento
df_procesado = procesar_datos(df)
df_procesado.head()

In [ ]:
df_procesado.info()

### 3.2 Convertir las categóricas restantes

Todavía tenemos Company, TypeName y OpSys como categoricas. Vamos a usar LabelEncoder porque tienen bastantes valores

In [ ]:
# guardamos los encoders para usarlos despues en test
le_company = LabelEncoder()
le_typename = LabelEncoder()
le_opsys = LabelEncoder()

df_procesado['Company_enc'] = le_company.fit_transform(df_procesado['Company'])
df_procesado['TypeName_enc'] = le_typename.fit_transform(df_procesado['TypeName'])
df_procesado['OpSys_enc'] = le_opsys.fit_transform(df_procesado['OpSys'])

In [ ]:
df_procesado.head()

### 2.3 Definir X e y

In [ ]:
# columnas que vamos a usar para el modelo
cols_modelo = [
    'Inches', 'Ram', 'Weight',
    'tiene_SSD', 'tiene_HDD', 'tiene_Flash', 'Capacidad_GB',
    'es_Touchscreen', 'es_IPS', 'Pixeles',
    'CPU_Intel', 'CPU_AMD', 'CPU_i3', 'CPU_i5', 'CPU_i7', 'CPU_GHz',
    'GPU_Nvidia', 'GPU_AMD', 'GPU_Intel',
    'es_Premium',
    'Company_enc', 'TypeName_enc', 'OpSys_enc'
]

X = df_procesado[cols_modelo]
y = df_procesado['Price_in_euros']

In [ ]:
X.shape

In [ ]:
y.shape

### 2.4 Dividir X_train, X_test, y_train, y_test

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
print("Train:", X_train.shape)
print("Validación:", X_val.shape)

## 4. Modelado

### 4.1 Baseline de modelos

In [ ]:
# probamos varios modelos con cross validation

modelos = {
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

print("Comparación de modelos (RMSE en cross-validation):")
print("="*50)

for nombre, modelo in modelos.items():
    # usamos neg_root_mean_squared_error porque sklearn minimiza
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
    rmse = -scores.mean()
    std = scores.std()
    print(f"{nombre}: {rmse:.2f} (+/- {std:.2f})")

Random Forest y Gradient Boosting dan los mejores resultados. Voy a probar a optimizar Random Forest

### 4.2 Sacar métricas, valorar los modelos

In [ ]:
# entrenamos Random Forest y evaluamos en validación
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# predicciones
y_pred_train = rf.predict(X_train)
y_pred_val = rf.predict(X_val)

# RMSE
rmse_train = root_mean_squared_error(y_train, y_pred_train)
rmse_val = root_mean_squared_error(y_val, y_pred_val)

print(f"RMSE Train: {rmse_train:.2f}")
print(f"RMSE Validación: {rmse_val:.2f}")

In [ ]:
# veamos la importancia de las features
importancias = pd.DataFrame({
    'feature': cols_modelo,
    'importancia': rf.feature_importances_
}).sort_values('importancia', ascending=False)

importancias.head(10)

In [ ]:
# gráfico de importancia
plt.figure(figsize=(10,6))
plt.barh(importancias['feature'][:10], importancias['importancia'][:10])
plt.xlabel('Importancia')
plt.title('Top 10 Features más importantes')
plt.gca().invert_yaxis()
plt.show()

Las variables mas importantes son la RAM, el tipo de portátil, los pixeles de la pantalla y la capacidad de almacenamiento. Tiene sentido!

### 4.3 Optimización

In [ ]:
# voy a hacer un GridSearchCV para encontrar los mejores parámetros
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor RMSE (CV):", -grid.best_score_)

In [ ]:
# usamos el mejor modelo
mejor_modelo = grid.best_estimator_

# evaluamos en validación
y_pred_val = mejor_modelo.predict(X_val)
rmse_val = root_mean_squared_error(y_val, y_pred_val)
print(f"RMSE Validación con mejor modelo: {rmse_val:.2f}")

### 4.4 Entrenar modelo final con todos los datos

Ahora que ya tenemos los mejores parámetros, entrenamos con TODOS los datos de train para tener el mejor modelo posible

In [ ]:
# entrenamos con todos los datos
model = RandomForestRegressor(
    n_estimators=grid.best_params_['n_estimators'],
    max_depth=grid.best_params_['max_depth'],
    min_samples_split=grid.best_params_['min_samples_split'],
    random_state=42
)

model.fit(X, y)
print("Modelo final entrenado!")

-----------------------------------------------------------------
## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.

In [ ]:
X_pred = pd.read_csv("./data/test.csv")
X_pred.head()

In [ ]:
X_pred.shape

In [ ]:
# guardamos los IDs para el submission
ids = X_pred['laptop_ID'].copy()

 ## 2. Replicar el procesado para ``test.csv``

In [ ]:
# aplicamos la misma función de procesamiento
X_pred_procesado = procesar_datos(X_pred)

In [ ]:
# aplicamos los mismos label encoders
# ojo: usamos transform, no fit_transform

# para Company, puede haber valores nuevos que no vimos en train
# en ese caso los ponemos como "desconocido"
def safe_transform(encoder, values):
    result = []
    for v in values:
        if v in encoder.classes_:
            result.append(encoder.transform([v])[0])
        else:
            # si no lo conocemos, ponemos -1
            result.append(-1)
    return result

X_pred_procesado['Company_enc'] = safe_transform(le_company, X_pred_procesado['Company'])
X_pred_procesado['TypeName_enc'] = safe_transform(le_typename, X_pred_procesado['TypeName'])
X_pred_procesado['OpSys_enc'] = safe_transform(le_opsys, X_pred_procesado['OpSys'])

In [ ]:
# seleccionamos las mismas columnas que usamos para entrenar
X_pred_final = X_pred_procesado[cols_modelo]

In [ ]:
X_pred_final.shape

In [ ]:
# hacemos las predicciones
predictions_submit = model.predict(X_pred_final)
predictions_submit[:10]

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [ ]:
sample = pd.read_csv("data/sample_submission.csv")

In [ ]:
sample.head()

In [ ]:
sample.shape

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [ ]:
submission = pd.DataFrame({
    'laptop_ID': ids,
    'Price_in_euros': predictions_submit
})

In [ ]:
submission.head()

In [ ]:
submission.shape

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [ ]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [ ]:
chequeador(submission)

## Notas finales

Cosas que he probado:
- Extraer características de las columnas de texto (CPU, GPU, Memory, etc)
- Probar varios modelos: Ridge, Lasso, Random Forest, Gradient Boosting
- Optimizar Random Forest con GridSearchCV

Cosas que se podrían mejorar:
- Hacer más feature engineering (extraer más info de la CPU, GPU)
- Probar XGBoost o LightGBM
- Hacer stacking de modelos
- Tratar outliers en el precio